# Create the Breast Cancer Classifier Artifact

Run this notebook inside the Triton Control code-server workspace. It trains a scikit-learn pipeline and writes `local_breast_cancer_repository/breast_cancer_classifier/1/model.joblib` for Triton's Python backend.

In [ ]:
%pip install scikit-learn joblib

: 

In [ ]:
from pathlib import Path

from joblib import dump
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
dataset = load_breast_cancer()
features = dataset.data.astype("float32")
labels = dataset.target.astype("int64")

x_train, x_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                class_weight="balanced",
            ),
        ),
    ]
)
model.fit(x_train, y_train)

predictions = model.predict(x_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Validation accuracy: {accuracy:.3f}")

In [ ]:
artifact_path = Path("local_breast_cancer_repository/breast_cancer_classifier/1/model.joblib")
artifact_path.parent.mkdir(parents=True, exist_ok=True)
dump(model, artifact_path)
print(f"Saved {artifact_path}")